[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/02_systems_and_evaluation/01_rag_failure_triage_lab.ipynb)

# Armory Notebook 4 — RAG Failure Triage Lab

| | |
|---|---|
| **Companion to** | Session 2, Lesson 1 — Defending the RAG Failure-Mode Analysis |
| **Runtime** | CPU only (no model downloads, no API keys) |
| **Estimated time** | 30 minutes |
| **Last verified** | 2026-07-14 |

This lab is a **working miniature of your instrumentation**. It builds a tiny corpus where *you* control ground truth, then runs the exact machinery your resume claim depends on: the three-way failure classifier, a chunk×k sweep, and the oracle-context condition.

The generator here is a **simulated** one with knobs you set (how often it ignores context, how often it hallucinates). That is deliberate: when you control the true failure rates, you can check whether your triage machinery *recovers* them. An instrument you have never tested against known ground truth is an instrument you cannot defend.

> [!IMPORTANT]
> Nothing here goes in the Metric Vault. These are synthetic mechanics, not your results. The transferable output is the **decision tree** and the discovery of where it misclassifies.

In [ ]:
%pip install -q "numpy>=1.26" "matplotlib>=3.8"

## Part 1 — A corpus with known gold labels

Ten questions, each answerable from exactly one gold passage. Distractor passages share vocabulary with the questions — the confusable neighbours that make retrieval non-trivial.

In [ ]:
import numpy as np

rng = np.random.default_rng(23)

# (question, gold_passage_text, answer) - a miniature ops-manual corpus
GOLD = [
    ("What pressure triggers the relief valve?",
     "The relief valve opens automatically once line pressure exceeds 12 bar.", "12 bar"),
    ("How long is the pump cooldown?",
     "Pump cooldown lasts 45 minutes before restart is permitted.", "45 minutes"),
    ("Which oil grade suits the gearbox?",
     "The gearbox requires ISO VG 220 oil in all climates.", "ISO VG 220"),
    ("What is the filter service interval?",
     "Replace the inlet filter every 500 operating hours.", "500 hours"),
    ("What voltage does the controller need?",
     "The controller runs on 24 V DC supplied by the cabinet rail.", "24 V DC"),
    ("What is the maximum ambient temperature?",
     "Do not operate the unit above 40 C ambient temperature.", "40 C"),
    ("How many bolts secure the flange?",
     "The flange is secured with eight M12 bolts torqued in a star pattern.", "eight"),
    ("What torque applies to the flange bolts?",
     "Torque each flange bolt to 95 Nm.", "95 Nm"),
    ("When must the seal be replaced?",
     "Replace the shaft seal at every second filter change.", "every second filter change"),
    ("What alarm code means low flow?",
     "Alarm code E14 indicates low flow through the primary loop.", "E14"),
]

# HARD distractors: two per question, deliberately reusing the question's keywords
# while containing NO answer. These are the confusable neighbours that make
# retrieval non-trivial - without them, a toy corpus hits recall 1.0 at k=2 and
# the whole failure taxonomy becomes unobservable.
HARD_DISTRACTORS = [
    "The relief valve assembly is inspected for pressure damage each quarter.",
    "Relief valve seats must be cleaned whenever line pressure readings drift.",
    "Pump cooldown procedures appear in the maintenance log appendix.",
    "The pump restart interlock is wired independently of cooldown timing.",
    "Gearbox oil samples are analysed to confirm the correct grade is in use.",
    "Oil grade labels on the gearbox housing fade in direct sunlight.",
    "Filter service records must list the interval and the technician name.",
    "The inlet filter service cart carries spare elements for each interval.",
    "Controller voltage alarms are logged whenever the supply rail dips.",
    "The controller cabinet houses the voltage regulator and the DC bus.",
    "Ambient temperature sensors are mounted away from the unit exhaust.",
    "Operating the unit at high ambient temperature shortens bearing life.",
    "Flange bolts are supplied pre-coated and must not be reused.",
    "The flange face is inspected before the bolts are secured.",
    "Torque values for the coupling bolts differ from the flange figures.",
    "A calibrated torque wrench is required for every bolt on the flange.",
    "Shaft seal leakage should be reported before the seal is replaced.",
    "The seal replacement kit is stocked alongside the filter elements.",
    "Alarm codes are cleared from the controller once flow is restored.",
    "Low flow through the primary loop can also trigger a vibration alarm.",
]

# Generic distractors: topical noise, low overlap with any question.
GENERIC_DISTRACTORS = [
    "Operators should wear hearing protection near the primary loop.",
    "The primary loop was extended during the 2021 refit.",
    "Vendor documentation is archived on the maintenance portal.",
    "Spare parts requests require a supervisor signature.",
    "Weekly inspections are recorded in the shift handover book.",
    "The workshop crane has a two tonne capacity.",
]

questions = [q for q, _, _ in GOLD]
passages  = [p for _, p, _ in GOLD] + HARD_DISTRACTORS + GENERIC_DISTRACTORS
gold_idx  = {qi: qi for qi in range(len(GOLD))}   # gold passage index == question index

print(f"{len(questions)} questions | {len(passages)} passages "
      f"({len(GOLD)} gold + {len(HARD_DISTRACTORS)} hard + {len(GENERIC_DISTRACTORS)} generic)")
print("\nThe hard distractors are the design decision that matters here: they share the")
print("question's vocabulary but never contain the answer, so the retriever must rank")
print("gold above near-neighbours. That is what produces retrieval-miss cases to study.")

## Part 2 — Chunking and a lexical retriever

A bag-of-words cosine retriever stands in for bge-small. It's weaker, which is *useful*: a weak retriever produces the retrieval-miss cases the taxonomy needs in order to be observable at all. (That's the same argument Lesson 1 makes for why bge-small was a defensible choice.)

Chunking splits each passage into `chunk_words`-word pieces — the mechanism that lets us watch **evidence fragmentation** appear.

In [ ]:
import re
from collections import Counter

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def chunk_corpus(passages: list[str], chunk_words: int) -> list[dict]:
    """Split each passage into fixed-size word chunks, remembering the source passage."""
    chunks = []
    for p_id, text in enumerate(passages):
        words = text.split()
        for start in range(0, len(words), chunk_words):
            piece = " ".join(words[start:start + chunk_words])
            chunks.append({"text": piece, "passage_id": p_id})
    return chunks

def embed(text: str, vocab: dict) -> np.ndarray:
    vec = np.zeros(len(vocab))
    for tok, n in Counter(tokenize(text)).items():
        if tok in vocab:
            vec[vocab[tok]] = n
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec      # L2-normalize -> inner product == cosine

def retrieve(question: str, chunks: list[dict], vocab: dict, k: int) -> list[int]:
    q_vec = embed(question, vocab)
    mat = np.stack([embed(c["text"], vocab) for c in chunks])
    scores = mat @ q_vec
    return list(np.argsort(-scores)[:k])

vocab = {t: i for i, t in enumerate(sorted({t for p in passages + questions for t in tokenize(p)}))}
print(f"vocab size: {len(vocab)}")
demo_chunks = chunk_corpus(passages, chunk_words=8)
print(f"chunks at chunk_words=8: {len(demo_chunks)}")
print("example chunk:", demo_chunks[0])

## Part 3 — A simulated generator with *known* failure behaviour

The knobs below are the ground truth your triage must recover:

- `p_ignore` — probability the generator answers from the wrong retrieved chunk despite having the gold (a **selection/grounding** failure).
- `p_hallucinate` — probability it invents an answer supported by nothing retrieved.
- `p_parametric` — probability it answers correctly **from memory** even when retrieval missed. This is the leakage knob from Lesson 1: it breaks the link between retrieval success and answer correctness.

In [ ]:
def simulate_generator(q_id, retrieved, chunks, *, p_ignore, p_hallucinate, p_parametric, rng):
    """Return (answer, provenance). provenance in {'gold','other_chunk','invented','memory'}."""
    gold_passage = gold_idx[q_id]
    gold_present = any(chunks[c]["passage_id"] == gold_passage for c in retrieved)
    correct = GOLD[q_id][2]

    if gold_present:
        if rng.random() < p_hallucinate:
            return "INVENTED-" + str(q_id), "invented"
        if rng.random() < p_ignore:
            return "WRONG-FROM-CHUNK-" + str(q_id), "other_chunk"
        return correct, "gold"

    # gold not retrieved: either recall from parametric memory, or fail
    if rng.random() < p_parametric:
        return correct, "memory"
    if rng.random() < p_hallucinate:
        return "INVENTED-" + str(q_id), "invented"
    return "WRONG-FROM-CHUNK-" + str(q_id), "other_chunk"

print("Generator knobs are the ground truth the triage classifier must recover.")

## Part 4 — The triage classifier (Lesson 1's decision tree, in code)

Order matters, and it is what makes the buckets mutually exclusive:

1. **gold in context?** No → `retrieval_miss`
2. answer supported by *some* retrieved chunk? Yes → `retrieved_but_ignored`
3. otherwise → `hallucination`

Note the classifier only sees what your real instrumentation sees: retrieved chunk IDs, the answer, and gold labels. It never sees `provenance` — that's the hidden truth we grade it against.

In [ ]:
def classify_failure(q_id, retrieved, chunks, answer) -> str:
    """Lesson 1's ordered decision tree. Sees only instrumented signals."""
    gold_passage = gold_idx[q_id]
    gold_present = any(chunks[c]["passage_id"] == gold_passage for c in retrieved)

    if not gold_present:
        return "retrieval_miss"
    # support test: is the answer traceable to any retrieved chunk?
    supported = answer.startswith("WRONG-FROM-CHUNK")   # stand-in for an NLI/overlap support check
    return "retrieved_but_ignored" if supported else "hallucination"

def run_eval(chunk_words, k, *, p_ignore=0.30, p_hallucinate=0.15, p_parametric=0.0, seed=0):
    rng_local = np.random.default_rng(seed)
    chunks = chunk_corpus(passages, chunk_words)
    rows = []
    for q_id, question in enumerate(questions):
        retrieved = retrieve(question, chunks, vocab, k)
        answer, provenance = simulate_generator(
            q_id, retrieved, chunks,
            p_ignore=p_ignore, p_hallucinate=p_hallucinate,
            p_parametric=p_parametric, rng=rng_local)
        is_correct = (answer == GOLD[q_id][2])
        gold_present = any(chunks[c]["passage_id"] == gold_idx[q_id] for c in retrieved)
        rows.append({
            "q_id": q_id, "correct": is_correct, "gold_present": gold_present,
            "provenance": provenance,
            "bucket": None if is_correct else classify_failure(q_id, retrieved, chunks, answer),
        })
    return rows

rows = run_eval(chunk_words=8, k=5)
acc = np.mean([r["correct"] for r in rows])
recall = np.mean([r["gold_present"] for r in rows])
buckets = Counter(r["bucket"] for r in rows if not r["correct"])
print(f"accuracy={acc:.2f}  recall@5={recall:.2f}")
print("failure buckets:", dict(buckets))

## Part 5 — The sweep: does recall move error?

This is the dissociation from Lesson 1's Refresher 2. Watch recall@k climb while accuracy stalls — the pattern that licenses "the binding constraint is downstream of retrieval."

In [ ]:
import matplotlib.pyplot as plt

INK, INK_SOFT = "#0b0b0b", "#52514e"
BLUE, AQUA, RED = "#2a78d6", "#1baf7a", "#e34948"

k_values = [1, 2, 3, 5, 8, 12, 20]
sweep = []
for k in k_values:
    reps = [run_eval(chunk_words=8, k=k, seed=s) for s in range(40)]   # average over generator randomness
    sweep.append({
        "k": k,
        "recall": np.mean([r["gold_present"] for rep in reps for r in rep]),
        "acc": np.mean([r["correct"] for rep in reps for r in rep]),
    })

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot([s["k"] for s in sweep], [s["recall"] for s in sweep], marker="o", color=BLUE,
        linewidth=2, markersize=7, label="recall@k (gold retrieved)")
ax.plot([s["k"] for s in sweep], [s["acc"] for s in sweep], marker="s", color=AQUA,
        linewidth=2, markersize=7, label="end-to-end accuracy")
ax.set_xlabel("retrieval depth k", color=INK_SOFT, fontsize=9)
ax.set_ylabel("rate", color=INK_SOFT, fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_title("The dissociation: recall rises, accuracy plateaus", fontsize=10, color=INK)
ax.legend(fontsize=8, frameon=False)
ax.tick_params(colors=INK_SOFT, labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e6e5e0", linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

for s in sweep:
    print(f"k={s['k']:>2}  recall={s['recall']:.2f}  accuracy={s['acc']:.2f}  gap={s['recall']-s['acc']:.2f}")
print("\nThe widening gap IS the argument. Evidence is present and not converted into answers.")

## Part 6 — Oracle context: the isolation experiment

Lesson 1, Follow-up 2's decisive move. Hand the generator **only** the gold passage. Whatever error remains has no retrieval explanation left — it is the generation floor.

In [ ]:
def run_oracle(*, p_ignore=0.30, p_hallucinate=0.15, seed=0):
    """Gold passage only, minimal context: retrieval cannot be blamed."""
    rng_local = np.random.default_rng(seed)
    chunks = [{"text": p, "passage_id": i} for i, p in enumerate(passages)]
    correct_flags = []
    for q_id in range(len(questions)):
        retrieved = [gold_idx[q_id]]                      # oracle: exactly the gold chunk
        answer, _ = simulate_generator(q_id, retrieved, chunks, p_ignore=p_ignore,
                                       p_hallucinate=p_hallucinate, p_parametric=0.0, rng=rng_local)
        correct_flags.append(answer == GOLD[q_id][2])
    return float(np.mean(correct_flags))

oracle_acc = np.mean([run_oracle(seed=s) for s in range(40)])
by_k = {s["k"]: s for s in sweep}

print(f"oracle-context accuracy (gold passage only, no retrieval) : {oracle_acc:.2f}")
print(f"=> GENERATION FLOOR: {1-oracle_acc:.0%} of questions fail even with PERFECT retrieval.\n")

print("Now decompose the error budget at two operating points:\n")
print(f"{'k':>3} {'recall':>7} {'accuracy':>9} {'retrieval share':>16} {'generation share':>17}")
for k in (3, 20):
    row = by_k[k]
    retrieval_share = max(0.0, oracle_acc - row["acc"])   # loss attributable to imperfect retrieval
    generation_share = 1 - oracle_acc                      # loss that survives perfect retrieval
    print(f"{k:>3} {row['recall']:>7.2f} {row['acc']:>9.2f} {retrieval_share:>16.2f} {generation_share:>17.2f}")

print("\nRead this the way a Bar Raiser will:")
print("  At k=3 recall is imperfect, so retrieval owns a real slice of the loss.")
print("  At k=20 recall is 1.00 - retrieval's share collapses to zero and EVERY")
print("  remaining error is generation-side. Same system, same generator; the")
print("  attribution changed entirely because the operating point changed.")
print("\nThat is why 'generation dominates' is only ever true AT AN OPERATING POINT.")
print("An unscoped version of that claim is the sentence you lose the round on.")

## Part 7 — Audit your own instrument

The point of controlling ground truth: check whether the triage classifier actually recovers it. Below, the hidden `provenance` is compared against the assigned bucket.

**Expected finding:** a `memory` answer (parametric leakage) that happens to be *wrong* gets bucketed by retrieval status alone — the classifier has no way to see that the generator never used the context. This is exactly the Lesson 1 limitation about leakage corrupting the taxonomy's denominators.

In [ ]:
# Leakage bites hardest where retrieval FAILS, so run at low k where misses are common.
K_LEAK = 1
print(f"At k={K_LEAK}, retrieval misses often - which is exactly where parametric memory hides.\n")

for p_par in (0.0, 0.5):
    reps = [run_eval(chunk_words=8, k=K_LEAK, p_parametric=p_par, seed=s) for s in range(60)]
    flat = [r for rep in reps for r in rep]
    acc = np.mean([r["correct"] for r in flat])
    recall = np.mean([r["gold_present"] for r in flat])
    missed = [r for r in flat if not r["gold_present"]]
    rescued = np.mean([r["correct"] for r in missed]) if missed else 0.0
    # what fraction of ALL failures does the taxonomy assign to retrieval_miss?
    fails = [r for r in flat if not r["correct"]]
    miss_share = np.mean([r["bucket"] == "retrieval_miss" for r in fails]) if fails else 0.0
    print(f"p_parametric={p_par:.1f} -> accuracy={acc:.3f}  recall@{K_LEAK}={recall:.2f}  "
          f"| missed-but-correct-from-memory: {rescued:5.1%}  "
          f"| share of failures blamed on retrieval: {miss_share:5.1%}")

print("\nRead the two rows carefully. recall is IDENTICAL - retrieval did not change at all.")
print("Yet accuracy rose, and the share of failures blamed on retrieval FELL. The pipeline")
print("looks better and retrieval looks less guilty, purely because the generator knew")
print("some answers already.")
print("\nThat is the confound: correctness stops implying retrieval worked, and wrongness")
print("stops implying it failed. Both taxonomy denominators shift. The control that")
print("detects it is the closed-book baseline - run the generator with NO context and")
print("see how much it already knows.")

## Close the loop

1. **Diff against your real pipeline.** Open your instrumentation code beside Part 4. Does your classifier order the checks the same way? Does it have a support test at all, or does it infer hallucination from *absence* of a match? Every difference is a gap — log it in [PROGRESS.md](../../PROGRESS.md).
2. **Name your leakage number.** Part 7 shows the mechanism; only your closed-book baseline gives the magnitude. If you never ran one, that is the honest gap to state in the assessment — *before* Follow-up 2 forces it out.
3. **assessment rehearsal.** Using the oracle decomposition in Part 6, say out loud in one minute: what the numbers license, what confounds remain, and what would falsify the conclusion. If you can't do it for this toy — where you *know* the ground truth — you can't do it for your project.

Nothing here fills a Metric Vault row. What it fills is the ability to answer *"how do you know your instrument works?"* with an experiment instead of an assurance.

## References & Credits

- Lewis et al. (2020), *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* — https://arxiv.org/abs/2005.11401
- Liu et al. (2023), *Lost in the Middle: How Language Models Use Long Contexts* — https://arxiv.org/abs/2307.03172
- NumPy (BSD-3), Matplotlib (PSF-based license).